# Vietnamese Hate Speech Detection — Fine-tuning PhoBERT on ViHSD

**Task**: 3-class classification of Vietnamese social-media comments — `CLEAN` (0), `OFFENSIVE` (1), `HATE` (2).

**Dataset**: [ViHSD](https://www.kaggle.com/datasets/leduc13/vihsd-data) (Luu et al., UIT-NLP) — 24,048 train / 2,672 dev / 6,680 test comments, heavily imbalanced (~83% clean, ~7% offensive, ~10% hate).

**Approach**: baseline (TF-IDF+LogReg) → PhoBERT fine-tuning with class-weighted cross-entropy → PhoBERT with focal loss → optional XLM-R comparison. Full rationale for every design choice is in `docs/research_notes.md` (PhoBERT paper, ViHSD paper, focal loss paper, Waseem & Hovy).

**Why this project**: content moderation is a real, deployed NLP problem, and Vietnamese is a lower-resource language where naive approaches (raw-syllable tokenization, plain accuracy) visibly fail — this notebook shows the fixes and quantifies their effect.

In [ ]:
# Kaggle sometimes assigns an older-generation GPU (e.g. Tesla P100, compute capability 6.0 /
# "sm_60") whose kernels the preinstalled PyTorch build no longer ships (recent builds only
# target sm_70+ — PyTorch dropped Maxwell/Pascal support around the 2.5 series). We need a torch
# version that BOTH still compiles sm_60 kernels (<2.5) AND has `torch.library.register_fake`,
# which this environment's transformers build requires (added in torch 2.4) — so 2.4.x is the
# version to pin. torchvision/torchaudio stay pinned to the original torch==2.10 build and then
# throw on mismatched compiled ops (`torchvision::nms does not exist`) — this notebook doesn't
# use either, so we just remove them rather than chase matching versions. Must all happen BEFORE
# torch is imported anywhere else in this notebook (a loaded CUDA extension can't be swapped out
# mid-process).
import subprocess, sys

def _gpu_compute_cap():
    try:
        out = subprocess.run(
            ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=10,
        )
        return out.stdout.strip().splitlines()[0] if out.returncode == 0 and out.stdout.strip() else None
    except Exception:
        return None

_cc = _gpu_compute_cap()
print('GPU compute capability:', _cc)

if _cc is not None and float(_cc) < 7.0:
    print(f'Compute capability {_cc} is older than sm_70 — the preinstalled torch build here '
          f'only ships sm_70+ kernels. Installing torch==2.4.1 (sm_60 support + register_fake)...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchvision', 'torchaudio'],
        check=True,
    )
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
         'torch==2.4.1', '--index-url', 'https://download.pytorch.org/whl/cu121'],
        check=True,
    )
    print('Reinstalled torch==2.4.1 (torchvision/torchaudio removed) for older-GPU compatibility.')

In [ ]:
import os, re, random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print('CUDA smoke-test:', (torch.zeros(1, device='cuda') + 1).item())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Load data

In [ ]:
import glob

candidates = glob.glob('/kaggle/input/**/train.csv', recursive=True)
print('/kaggle/input contents:', os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'MISSING')
print('train.csv candidates found:', candidates)
assert candidates, 'ViHSD dataset not attached to this kernel — check dataset_sources in kernel-metadata.json'
DATA_DIR = os.path.dirname(candidates[0])
print('Using DATA_DIR =', DATA_DIR)
LABEL_NAMES = {0: 'CLEAN', 1: 'OFFENSIVE', 2: 'HATE'}

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
dev_df   = pd.read_csv(f'{DATA_DIR}/dev.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

for name, df in [('train', train_df), ('dev', dev_df), ('test', test_df)]:
    print(name, df.shape, dict(df['label_id'].value_counts().sort_index()))

train_df.head()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5,3))
counts = train_df['label_id'].value_counts().sort_index()
ax.bar([LABEL_NAMES[i] for i in counts.index], counts.values, color=['#4C72B0','#DD8452','#C44E52'])
ax.set_title('Train label distribution (ViHSD)')
ax.set_ylabel('count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, str(v), ha='center')
plt.tight_layout(); plt.show()

## 2. Text cleaning

Light cleaning only — strip URLs/mentions/extra whitespace, keep diacritics and punctuation (they carry sentiment/emphasis signal in Vietnamese social text, e.g. repeated punctuation, emoji-adjacent slang).

In [ ]:
URL_RE = re.compile(r'https?://\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
WS_RE = re.compile(r'\s+')

def clean_text(s: str) -> str:
    s = str(s)
    s = URL_RE.sub(' ', s)
    s = MENTION_RE.sub(' ', s)
    s = WS_RE.sub(' ', s).strip()
    return s

for df in (train_df, dev_df, test_df):
    df['text_clean'] = df['free_text'].apply(clean_text)

train_df[['free_text', 'text_clean']].head()

## 3. Stage 1 — Baseline: TF-IDF + Logistic Regression

Gives a reference point to quantify how much PhoBERT actually buys us, and what a "good enough with 5 minutes of effort" solution looks like.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, confusion_matrix

tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=30000)
Xtr = tfidf.fit_transform(train_df['text_clean'])
Xte = tfidf.transform(test_df['text_clean'])
ytr, yte = train_df['label_id'], test_df['label_id']

clf = LogisticRegression(max_iter=2000, class_weight='balanced', C=2.0)
clf.fit(Xtr, ytr)
pred_baseline = clf.predict(Xte)

baseline_macro_f1 = f1_score(yte, pred_baseline, average='macro')
print('Baseline macro-F1:', round(baseline_macro_f1, 4))
print(classification_report(yte, pred_baseline, target_names=list(LABEL_NAMES.values()), digits=3))

## 4. Stage 2 — PhoBERT fine-tuning (class-weighted cross-entropy)

PhoBERT's pretraining corpus was word-segmented (multi-syllable Vietnamese words joined with `_`), so we segment with `pyvi` before tokenizing — feeding it raw syllables measurably degrades subword quality since the vocabulary expects segmented tokens (see `docs/research_notes.md` for the PhoBERT paper reference).

In [ ]:
!pip install -q pyvi

In [ ]:
from pyvi import ViTokenizer

def segment(s: str) -> str:
    try:
        return ViTokenizer.tokenize(s)
    except Exception:
        return s

for df in (train_df, dev_df, test_df):
    df['text_seg'] = df['text_clean'].apply(segment)

print(train_df.loc[0, 'text_clean'])
print(train_df.loc[0, 'text_seg'])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = 'vinai/phobert-base'
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

class ViHSDDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tokenizer(list(texts), truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item['labels'] = self.labels[idx]
        return item

train_ds = ViHSDDataset(train_df['text_seg'], train_df['label_id'])
dev_ds   = ViHSDDataset(dev_df['text_seg'], dev_df['label_id'])
test_ds  = ViHSDDataset(test_df['text_seg'], test_df['label_id'])
len(train_ds), len(dev_ds), len(test_ds)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2])
class_weights = compute_class_weight('balanced', classes=classes, y=train_df['label_id'].values)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print('class weights:', class_weights)

# use_safetensors=True: transformers refuses torch.load-based checkpoints unless torch>=2.6
# (a CVE-driven restriction), but torch>=2.6 dropped sm_60 (Pascal/P100) support — so we sidestep
# torch.load entirely by requiring the safetensors checkpoint format instead.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, use_safetensors=True).to(DEVICE)

In [ ]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': (preds == labels).mean(),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
    }

# GPU-quota may be exhausted for the week (Kaggle silently falls back to CPU) — keep runtime
# bounded in that case rather than letting a 4-epoch PhoBERT-base run stall for hours.
if torch.cuda.is_available():
    NUM_EPOCHS, TRAIN_BS, EVAL_BS = 4, 32, 64
else:
    print('WARNING: no GPU available this run — using reduced schedule to keep CPU runtime bounded.')
    NUM_EPOCHS, TRAIN_BS, EVAL_BS = 2, 16, 32

args = TrainingArguments(
    output_dir='/kaggle/working/phobert-vihsd',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to='none',
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## 5. Stage 2 evaluation

In [ ]:
test_output = trainer.predict(test_ds)
test_logits, test_labels = test_output.predictions, test_output.label_ids
test_preds = np.argmax(test_logits, axis=-1)

phobert_macro_f1 = f1_score(test_labels, test_preds, average='macro')
print('PhoBERT macro-F1 (test):', round(phobert_macro_f1, 4))
print('Baseline macro-F1 (test):', round(baseline_macro_f1, 4))
print(classification_report(test_labels, test_preds, target_names=list(LABEL_NAMES.values()), digits=3))

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(4,4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_xticklabels(LABEL_NAMES.values())
ax.set_yticks(range(3)); ax.set_yticklabels(LABEL_NAMES.values())
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
ax.set_title('PhoBERT confusion matrix (test)')
plt.tight_layout(); plt.show()

## 6. Stage 2 error analysis

Focus on the hardest boundary: OFFENSIVE vs HATE. These differ mainly by *target and intent* (general profanity vs. targeted attack on a group/individual), which is exactly the kind of pragmatic distinction that surface-level features struggle with (see `docs/research_notes.md` — Waseem & Hovy, ViHSD paper).

In [ ]:
errors = test_df.copy()
errors['pred'] = test_preds
errors['true'] = test_labels
mis = errors[errors['pred'] != errors['true']]

off_hate_confusions = mis[mis['true'].isin([1,2]) & mis['pred'].isin([1,2])]
print(f'{len(mis)} total misclassifications on test set ({len(mis)/len(errors):.1%})')
print(f'{len(off_hate_confusions)} of which are OFFENSIVE<->HATE confusions')

pd.set_option('display.max_colwidth', 120)
off_hate_confusions[['free_text','true','pred']].assign(
    true=lambda d: d['true'].map(LABEL_NAMES), pred=lambda d: d['pred'].map(LABEL_NAMES)
).head(15)

## 7. Stage 3 — Focal loss (alternative imbalance handling)

Class-weighted CE (Stage 2) rescales the *loss per class*. Focal loss (Lin et al. 2017) instead down-weights *easy, already-confident* examples via a `(1 - p_t)^γ` factor, regardless of class — a different mechanism for the same imbalance problem. We fine-tune a fresh PhoBERT with focal loss and compare macro-F1 against Stage 2 rather than assuming either wins a priori.

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def _pt(self, logits, labels):
        # p_t must reflect the model's TRUE confidence, independent of class weight --
        # computing it from a weighted CE (the original bug here) silently distorts it to
        # p_true ** weight, which for the majority class (weight ~ 0.4) pushes p_t toward 1
        # and kills its gradient almost entirely (see docs/research_notes.md).
        ce_unweighted = nn.functional.cross_entropy(logits, labels, weight=None, reduction='none')
        return torch.exp(-ce_unweighted)

    def forward(self, logits, labels):
        ce = nn.functional.cross_entropy(logits, labels, weight=self.weight, reduction='none')
        pt = self._pt(logits, labels)
        return ((1 - pt) ** self.gamma * ce).mean()


# Sanity checks (guard against the bug above regressing silently again):
# 1) gamma=0 must reduce exactly to the (weighted) CrossEntropyLoss.
# 2) p_t must be identical regardless of `weight` -- it must not depend on class weighting.
_sanity_logits = torch.randn(16, 3)
_sanity_labels = torch.randint(0, 3, (16,))
_sanity_weight = torch.tensor([0.4, 3.0, 5.0])

_fl_gamma0 = FocalLoss(gamma=0.0, weight=_sanity_weight)(_sanity_logits, _sanity_labels)
_ce_ref = nn.functional.cross_entropy(_sanity_logits, _sanity_labels, weight=_sanity_weight, reduction='none').mean()
assert torch.allclose(_fl_gamma0, _ce_ref, atol=1e-5), 'FocalLoss(gamma=0) must equal weighted CrossEntropyLoss'

_pt_weighted = FocalLoss(weight=_sanity_weight)._pt(_sanity_logits, _sanity_labels)
_pt_unweighted = FocalLoss(weight=None)._pt(_sanity_logits, _sanity_labels)
assert torch.allclose(_pt_weighted, _pt_unweighted), 'p_t must not depend on class weight'
print('FocalLoss sanity checks passed: gamma=0 == weighted CE, p_t independent of weight.')


focal_loss_fn = FocalLoss(gamma=2.0, weight=class_weights)

class FocalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss = focal_loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


model_focal = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, use_safetensors=True).to(DEVICE)

args_focal = TrainingArguments(
    output_dir='/kaggle/working/phobert-vihsd-focal',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to='none',
    seed=SEED,
)

trainer_focal = FocalTrainer(
    model=model_focal,
    args=args_focal,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    compute_metrics=compute_metrics,
)

trainer_focal.train()

In [ ]:
focal_test_output = trainer_focal.predict(test_ds)
focal_logits, _ = focal_test_output.predictions, focal_test_output.label_ids
focal_preds = np.argmax(focal_logits, axis=-1)

focal_macro_f1 = f1_score(test_labels, focal_preds, average='macro')
print('Focal-loss PhoBERT macro-F1 (test):', round(focal_macro_f1, 4))
print(classification_report(test_labels, focal_preds, target_names=list(LABEL_NAMES.values()), digits=3))

## 8. Stage 4 — XLM-R-base comparison (optional, GPU-only)

The ViHSD paper reports XLM-R-base as roughly on par with PhoBERT on this task. We run it only when a GPU is available, to keep the notebook's CPU-fallback runtime bounded — this stage is a nice-to-have comparison point, not required for the core deliverable.

In [ ]:
xlmr_macro_f1 = None

if torch.cuda.is_available():
    XLMR_NAME = 'xlm-roberta-base'
    xlmr_tokenizer = AutoTokenizer.from_pretrained(XLMR_NAME)

    class XLMRDataset(torch.utils.data.Dataset):
        def __init__(self, texts, labels):
            self.enc = xlmr_tokenizer(list(texts), truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors='pt')
            self.labels = torch.tensor(list(labels), dtype=torch.long)
        def __len__(self):
            return len(self.labels)
        def __getitem__(self, idx):
            item = {k: v[idx] for k, v in self.enc.items()}
            item['labels'] = self.labels[idx]
            return item

    # XLM-R was NOT pretrained on word-segmented Vietnamese, so we feed it text_clean (unsegmented),
    # unlike PhoBERT — segmenting would only hurt it here.
    xlmr_train_ds = XLMRDataset(train_df['text_clean'], train_df['label_id'])
    xlmr_dev_ds   = XLMRDataset(dev_df['text_clean'], dev_df['label_id'])
    xlmr_test_ds  = XLMRDataset(test_df['text_clean'], test_df['label_id'])

    model_xlmr = AutoModelForSequenceClassification.from_pretrained(XLMR_NAME, num_labels=3, use_safetensors=True).to(DEVICE)

    args_xlmr = TrainingArguments(
        output_dir='/kaggle/working/xlmr-vihsd',
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        greater_is_better=True,
        fp16=True,
        logging_steps=100,
        report_to='none',
        seed=SEED,
    )

    trainer_xlmr = WeightedTrainer(
        model=model_xlmr,
        args=args_xlmr,
        train_dataset=xlmr_train_ds,
        eval_dataset=xlmr_dev_ds,
        compute_metrics=compute_metrics,
    )
    trainer_xlmr.train()

    xlmr_test_output = trainer_xlmr.predict(xlmr_test_ds)
    xlmr_preds = np.argmax(xlmr_test_output.predictions, axis=-1)
    xlmr_macro_f1 = f1_score(test_labels, xlmr_preds, average='macro')
    print('XLM-R macro-F1 (test):', round(xlmr_macro_f1, 4))
    print(classification_report(test_labels, xlmr_preds, target_names=list(LABEL_NAMES.values()), digits=3))
else:
    print('Skipping Stage 4 (XLM-R) — no GPU available this run.')

## 9. Model selection & save

Pick whichever stage scored highest macro-F1 on test, save that one as the deliverable checkpoint (this is what `backend/main.py` loads when `MODEL_BACKEND=phobert`).

In [ ]:
candidates = {
    'phobert_weighted_ce': (phobert_macro_f1, trainer, tokenizer),
    'phobert_focal': (focal_macro_f1, trainer_focal, tokenizer),
}
if xlmr_macro_f1 is not None:
    candidates['xlmr_weighted_ce'] = (xlmr_macro_f1, trainer_xlmr, xlmr_tokenizer)

best_name, (best_f1, best_trainer, best_tokenizer) = max(candidates.items(), key=lambda kv: kv[1][0])
print('Stage comparison (test macro-F1):')
for name, (f1, *_ ) in candidates.items():
    marker = ' <-- best' if name == best_name else ''
    print(f'  {name}: {f1:.4f}{marker}')

SAVE_DIR = '/kaggle/working/phobert-vihsd-final'
best_trainer.save_model(SAVE_DIR)
best_tokenizer.save_pretrained(SAVE_DIR)
print(f'\nSaved best model ({best_name}, macro-F1={best_f1:.4f}) to {SAVE_DIR}')

## 10. Summary

| Stage | Model | Macro-F1 (test) |
|---|---|---|
| 1 | TF-IDF + Logistic Regression | see Stage 1 output above |
| 2 | PhoBERT-base, word-segmented, class-weighted CE | see Stage 2 output above |
| 3 | PhoBERT-base, word-segmented, focal loss | see Stage 3 output above |
| 4 | XLM-R-base, class-weighted CE (GPU runs only) | see Stage 4 output above (skipped on CPU) |

Full reasoning behind each design choice is in `docs/research_notes.md` (PhoBERT, ViHSD, focal loss, Waseem & Hovy references).

**Limitations / future work**:
- OFFENSIVE vs HATE remains the dominant error mode — could be improved with more context (thread-level, not single-comment) or a larger/augmented HATE sample (it's the rarest class).
- No slang/teencode-normalization step (e.g. `mk` → `mày`, elongated letters) — worth testing whether normalizing before segmentation helps.
- Single random seed reported per stage; a rigorous follow-up would run 3 seeds and report mean ± std, especially for the rare HATE class.
- If Stage 4 (XLM-R) turns out best, note that `backend/main.py`'s PhoBERT-loading path (which includes the `pyvi` segmentation step) would need a small adjustment — XLM-R wasn't pretrained on segmented Vietnamese, so it should be served on `text_clean` directly, not `text_seg`.

## 11. Ensemble artifacts — persist per-model probabilities (dev + test)

All 4 stages are trained in this same kernel run, but none of their predicted probabilities
were ever written to disk — only the single "best" checkpoint gets saved, so any ensembling
work would otherwise require re-running inference in a separate session. Instead we dump
compact per-class-probability CSVs (no model weights) for baseline / PhoBERT-CE /
PhoBERT-focal / XLM-R on both `dev` and `test`, reusing the trainers/models still in memory
from the stages above. These feed `Final/scripts/build_ensemble.py` locally.

In [ ]:
def save_probs(name, probs, split_name, df):
    out = pd.DataFrame(probs, columns=[f'p_{LABEL_NAMES[i]}' for i in range(3)])
    out['label_id'] = df['label_id'].values
    out['text_clean'] = df['text_clean'].values
    path = f'/kaggle/working/probs_{name}_{split_name}.csv'
    out.to_csv(path, index=False)
    print('Saved', path, out.shape)


def softmax_np(logits):
    return torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()


# Baseline (TF-IDF + LogReg) -- needs a fresh transform of dev (only test was transformed above)
Xdev_baseline = tfidf.transform(dev_df['text_clean'])
save_probs('baseline', clf.predict_proba(Xdev_baseline), 'dev', dev_df)
save_probs('baseline', clf.predict_proba(Xte), 'test', test_df)

# PhoBERT class-weighted CE
dev_output_ce = trainer.predict(dev_ds)
save_probs('phobert_ce', softmax_np(dev_output_ce.predictions), 'dev', dev_df)
save_probs('phobert_ce', softmax_np(test_logits), 'test', test_df)

# PhoBERT focal loss
dev_output_focal = trainer_focal.predict(dev_ds)
save_probs('phobert_focal', softmax_np(dev_output_focal.predictions), 'dev', dev_df)
save_probs('phobert_focal', softmax_np(focal_logits), 'test', test_df)

# XLM-R (GPU-only stage -- skip cleanly if this run had no GPU)
if xlmr_macro_f1 is not None:
    xlmr_dev_output = trainer_xlmr.predict(xlmr_dev_ds)
    save_probs('xlmr', softmax_np(xlmr_dev_output.predictions), 'dev', dev_df)
    save_probs('xlmr', softmax_np(xlmr_test_output.predictions), 'test', test_df)
else:
    print('Skipping XLM-R prob dump -- Stage 4 was skipped (no GPU this run).')

print('\nAll probability artifacts saved under /kaggle/working/ as small CSVs (no weights) -- '
      'download via `kaggle kernels output`.')